In [ ]:
# In this workbook, the experiment results from insurance_performance_eval.py are read and evaluated
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import os
from ray import tune
import torch
from sklearn.model_selection import KFold
from lidglm import ExtendedGLM_Utils
from lidglm.extended_glm.extended_glm_eval_utls import Extended_glm_eval_utils
from lidglm.application_utilities.performance_analysis import train_set_split
from sklearn.model_selection import train_test_split
from lidglm.application_utilities.performance_analysis import normalize_covariates
import seaborn as sns
import json
output_dir = "./outputs/"

In [ ]:
# The following have to be adjusted to point to the folders with the experiment results:
insurance_result_dir = "D:/Insurance"
ldglm_simulation_result_subdir = insurance_result_dir + "/09-01"
sddr_simulation_result_subdir = insurance_result_dir + "/09-01"
ldglm_with_nu_d_simulation_result_subdir = insurance_result_dir + "/09-02"
ldglm_session_subdir = ldglm_simulation_result_subdir + "/session_2026-09-01_13-36-22_515956_62584"
sddr_session_subdir = ldglm_session_subdir
ldglm_with_nu_d_session_subdir = ldglm_with_nu_d_simulation_result_subdir + "/session_2026-09-02_22-07-01_036386_65660"

# set directories from the ones defined above; should not change between simulations
artifact_dir = ldglm_session_subdir + "/artifacts"
artifact_dir_with_nu_d = ldglm_with_nu_d_session_subdir + "/artifacts"

In [ ]:
# load separate result dataframes:
lid_glm_metrics_dataframe_no_nu_d = torch.load(ldglm_simulation_result_subdir + "/k_fold_performance_analysis_insurance09-01_lidglm_metrics_dataframe.pt")
lid_glm_metrics_dataframe_with_nu_d = torch.load(ldglm_with_nu_d_simulation_result_subdir + "/k_fold_performance_analysis_insurance_normalized09-03_lidglm_with_nu_d_metrics_dataframe.pt")
traditional_glm_metrics_df = torch.load(ldglm_simulation_result_subdir + "/k_fold_performance_analysis_insurance09-01_traditional_metrics_dataframe.pt")
elastic_net_metrics_df = torch.load(ldglm_simulation_result_subdir + "/k_fold_performance_analysis_insurance09-01_elastic_net_metrics_dataframe.pt")
sddr_metrics_df = torch.load(sddr_simulation_result_subdir + "/k_fold_performance_analysis_insurance09-01_custom_sddr_metrics_dataframe.pt")

# we rename the columns called "test loglike" and "test mse" to indicate which model they belong to:
for df, suffix in [(lid_glm_metrics_dataframe_no_nu_d, "_lid_glm_no_nu_d"), (lid_glm_metrics_dataframe_with_nu_d, "_lid_glm_with_nu_d"), 
                   (traditional_glm_metrics_df, "_traditional"), (elastic_net_metrics_df, "_elastic_net"), (sddr_metrics_df, "_sddr")]:
    df.rename(columns={"test loglike": "test loglike" + suffix, "test mse": "test mse" + suffix}, inplace=True)

# join the dataframes by the "splits" index column:
all_metrics_df = lid_glm_metrics_dataframe_with_nu_d.merge(traditional_glm_metrics_df, on="split", suffixes=("_lid_glm_with_nu_d", "_traditional"))
all_metrics_df = all_metrics_df.merge(sddr_metrics_df, on="split", suffixes=("", "_sddr"))
all_metrics_df = all_metrics_df.merge(lid_glm_metrics_dataframe_no_nu_d, on="split", suffixes=("", "_lid_glm_no_nu_d"))
all_metrics_df = all_metrics_df.merge(elastic_net_metrics_df, on="split", suffixes=("", "_elastic_net"))

print(all_metrics_df)
all_metrics_df.to_csv(output_dir + "/insurance_all_models_performance_comparison_metrics_dataframe.csv")

In [ ]:
# First, we will vizualize the performance across the different models:
fig, ax = plt.subplots(figsize=(4, 5.25))
models = [ 'GLM', 'LID-GLM \n no $T_d$', "LiD-GLM \n with $T_d$",'SDDR']
# we change the sign to obtain the negative loglike:
test_loglikes = [all_metrics_df['test loglike_traditional']*(-1), 
                 all_metrics_df['test loglike_lid_glm_no_nu_d']*(-1), 
                 all_metrics_df['test loglike_lid_glm_with_nu_d']*(-1),
                 all_metrics_df['test loglike_sddr']*(-1)]
box = ax.boxplot(test_loglikes)
ax.set_ylabel('Test negative Log-Likelihood', fontsize=12)
ax.set_xticklabels(models, fontsize = 12)

for median in box['medians']:
    median.set(color='red', linewidth=1.5, linestyle="-")

plt.tight_layout()
fig.savefig(output_dir + "/insurance_model_performance_comparison_loglike.pdf")

In [ ]:
# we construct a table aggregating mean and std of the test log-likelihoods for each model:
summary_table = pd.DataFrame({
    "Model": models,
    "Mean Test Log-Likelihood": [all_metrics_df["test loglike_traditional"].mean(),
                                all_metrics_df["test loglike_lid_glm_no_nu_d"].mean(),
                                all_metrics_df["test loglike_lid_glm_with_nu_d"].mean(),
                                all_metrics_df["test loglike_sddr"].mean()],
    "Std Test Log-Likelihood": [all_metrics_df["test loglike_traditional"].std(),
                               all_metrics_df["test loglike_lid_glm_no_nu_d"].std(),
                               all_metrics_df["test loglike_lid_glm_with_nu_d"].std(),
                               all_metrics_df["test loglike_sddr"].std()]
})
print(summary_table)
summary_table.to_csv(ldglm_simulation_result_subdir + "/insurance_test_loglike_summary_table.csv", index=False)

In [ ]:
##################import and preprocess dataset:#####################

torch.set_num_threads(1)
#Dataset loading and preprocessing:

# fetch dataset 
insurance_data = pd.read_csv("../Datasets/insurance_data/insurance.csv")

# the sex and smoker variables are strings but should be binary
# sex is coded as 'female', 'male' and smoker as 'yes', 'no'
# We'll recode female=1, male=0 and yes=1, no=0

insurance_data["sex"] = insurance_data["sex"].replace({"female":1, "male":0}).astype(int)
insurance_data["smoker"] = insurance_data["smoker"].replace({"yes":1, "no":0}).astype(int)

# region is a categorical variable with categories 'southwest', 'southeast', 'northwest', 'northeast'
# we'll do a dummy encoding and delete the reference
insurance_data = pd.get_dummies(insurance_data, columns=["region"], drop_first=True, dtype=int)

#finally, we pop the target column
target = insurance_data.pop("charges")
# improve inversion stability: normalize the target
target = (target-target.mean())/target.std()
target = np.array(target)
covariates = np.array(insurance_data)
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
util = ExtendedGLM_Utils()
eval_util = Extended_glm_eval_utils()
n_covariates = covariates.shape[1]
X= insurance_data


In [ ]:
# For each split we will store the beta coefficients for the top n runs (measured by their validation loss in the result grid) both with and without PHO:
model_name = "ldglm"
artifact_dir_used = artifact_dir if model_name == "ldglm" else artifact_dir_with_nu_d
simulation_dir_used = ldglm_simulation_result_subdir if model_name == "ldglm" else ldglm_with_nu_d_simulation_result_subdir
read_top_n = 20

# initialize the dataframes:
beta_dataframe_top_n_runs = pd.DataFrame(columns = ["split", "type", "model_ranking", "bias"] + X.columns.tolist())
beta_dataframe_top_n_runs.set_index(["split", "type", "model_ranking"], inplace=True)

# We want to check the dependence of model performance on the allowed Lipschitz constant
# For this, we read all models from all splits, pool them by their allowed Lipschitz constant and do boxplots.
lipschitz_validation_performance_dict = {}
lipschitz_test_performance_dict = {}

beta_differences_dataframe = pd.DataFrame(columns = ["split", "type","bias"] + X.columns.tolist())
beta_differences_dataframe.set_index(["split", "type"], inplace=True)
best_results_metrics = []
best_results_configs= []

all_tested_model_configs = {}

# We iterate through all splits
for i, (train_index, test_index) in enumerate(kf.split(target)):
    actual_train_index, val_index = train_test_split(train_index, test_size=0.2, random_state=42)
    covariates_used = normalize_covariates(covariates, train_index)
    X_train_val, X_test = covariates_used[train_index, :], covariates_used[test_index, :]
    y_train_val, y_test = target[train_index], target[test_index]

    # load the result grid object for this split:
    result_grid = tune.Tuner.restore(os.path.join(simulation_dir_used, "k_fold_performance_analysis_insurance_"+ model_name + "_split_"+str(i+1)), trainable =train_set_split).get_results()


    # We iterate over the top n runs to store their beta coefficients both with and without PHO:
    result_index = 0
    for run in result_grid.get_dataframe(filter_metric="validation_loss", filter_mode = "min").sort_values(by="validation_loss", ascending=True).head(read_top_n).iterrows():
        run_id = run[0]
        # load the extended GLM model from the checkpoint:
        loaded_run_model = util.load_from_ray_checkpoint(result_grid[run_id].get_best_checkpoint(metric="validation_loss", mode="min"), artifact_dir=artifact_dir_used).eval()
        # read the beta coefficient vector without PHO and append to the dataframe:
        beta_no_pho = np.append(loaded_run_model.read_linear_bias(), loaded_run_model.read_linear_weights())
        beta_dataframe_top_n_runs.loc[(i, "no PHO", result_index),:] = beta_no_pho
        # read the beta coefficient vector with PHO and append to the dataframe:
        beta_with_pho = eval_util.custom_PHO(loaded_run_model, torch.tensor(X_train_val, dtype=torch.float32))[0]
        beta_dataframe_top_n_runs.loc[(i, "with PHO", result_index),:] = beta_with_pho
        result_index += 1

    # We want to check the dependence of model performance on the allowed Lipschitz constant.
    # We pool all splits together for this and consider all models from all splits - not just the best ones.
    
    best_result_df = result_grid.get_dataframe(filter_metric="validation_loss", filter_mode = "min")
    for lipschitz_constant in best_result_df["config/model_specifications/lipschitz_const"]:
        if lipschitz_constant not in lipschitz_validation_performance_dict.keys():
            lipschitz_validation_performance_dict[lipschitz_constant] = best_result_df[best_result_df["config/model_specifications/lipschitz_const"] == lipschitz_constant]["validation_loss"].to_list()
        else:
            lipschitz_validation_performance_dict[lipschitz_constant] = lipschitz_validation_performance_dict[lipschitz_constant] + \
                best_result_df[best_result_df["config/model_specifications/lipschitz_const"] == lipschitz_constant]["validation_loss"].to_list()

    # To make our plot more readable later, we only show the top 10 models in each split; also, we list test performance instead of validation loss.
    top_n_per_const = 10
    for key in lipschitz_validation_performance_dict.keys():
        if key not in lipschitz_test_performance_dict.keys():
            lipschitz_test_performance_dict[key] = []
        filtered_df = best_result_df[best_result_df["config/model_specifications/lipschitz_const"] == key].sort_values(by="validation_loss", ascending=True).head(top_n_per_const)
        for run_id, run in filtered_df.iterrows():
            # load the extended GLM model from the checkpoint:
            loaded_run_model = util.load_from_ray_checkpoint(result_grid[run_id].get_best_checkpoint(metric="validation_loss", mode="min"), artifact_dir=artifact_dir_used).eval()
            # compute test log-likelihood:
            test_log_like = loaded_run_model.log_like_obs(exog = torch.tensor(X_test, dtype=torch.float32),endog= torch.tensor(y_test, dtype=torch.float32)).detach().cpu().numpy().mean()
            lipschitz_test_performance_dict[key].append(-test_log_like.item()) #we flip the sign to obtain the negative loglike


    # for each split, we want to find the hyperparameters of the best performing LiD-GLM model and also re-compute its performance
    best_result = result_grid.get_best_result(metric="validation_loss", mode="min", scope = "all")
    best_results_metrics.append(best_result.metrics)
    best_results_configs.append(best_result.config["model_specifications"])

    # we also list all model configurations that were tested during hyperparameter optimization:
    if len(all_tested_model_configs.keys()) == 0:
        for key in best_result.config["model_specifications"].keys():
            all_tested_model_configs[key] = set([])

    for key in all_tested_model_configs.keys():
        all_tested_model_configs[key] = all_tested_model_configs[key].union(set(best_result_df["config/model_specifications/"+key].unique()))

In [ ]:
# we convert the data types from numpy to base Python types and also round to 3 digits
for key, config_set in all_tested_model_configs.items():
    for item in config_set:
        if isinstance(item, np.integer):
            config_set.remove(item)
            config_set.add(int(item))
        elif isinstance(item, np.floating):
            config_set.remove(item)
            config_set.add(round(float(item), 3))

with open(output_dir + f"/insurance_all_tested_model_hyperparameter_configs_{model_name}.json", "w") as f:
    json.dump({key: list(value) for key, value in all_tested_model_configs.items()}, f)

print(all_tested_model_configs)

In [ ]:
# we show the chosen hyperparameter configuarions in each split:
best_result_df = pd.DataFrame(best_results_configs)
pd.concat((best_result_df, pd.DataFrame(best_results_metrics)), axis = 1)
print(best_result_df)
best_result_df.to_csv(output_dir + "/insurance_best_hyperparams_"+ model_name + ".csv")

In [ ]:
# we do vertical boxplots of the estimated coefficients with/without PHO with separate subplots for each covariate
# in each subplot, we have two groups of plots: with/without PHO, each group has 5 plots, colored by split
n_covariates_per_row = 3
n_rows = math.ceil(n_covariates / n_covariates_per_row)
fig, axs = plt.subplots(n_rows, n_covariates_per_row, figsize = (2.3*(n_covariates_per_row+1), 3*n_rows))
# we have a prime number of covariates, so some subplots will be empty and deleted:
number_of_empty_plots = n_covariates_per_row * n_rows - n_covariates
for empty_plot_index in range(number_of_empty_plots):
    fig.delaxes(axs[-1, -1 - empty_plot_index])

seaborn_df = pd.melt(beta_dataframe_top_n_runs.reset_index(), id_vars=["split", "type", "model_ranking"], value_vars=X.columns.tolist(), var_name="covariate", value_name="coefficient")
seaborn_df = seaborn_df[seaborn_df["split"]<= 4] # we only consider the first 5 splits to avoid overcrowding the plot

covariates = X.columns.tolist()
# we rename the types to PHO and No PHO for better readability in the plot
seaborn_df["type"] = seaborn_df["type"].replace({"no PHO": "No PHO", "with PHO": "PHO"})
# we rename the column "split" to "Split" for better readability in the plot
seaborn_df.columns = seaborn_df.columns.str.replace("split", "Split")

for i in range(n_covariates):
    used_dataframe = seaborn_df[seaborn_df["covariate"]==covariates[i]]
    legend = False if i < n_covariates-1  else "auto"
    ax = axs[i//n_covariates_per_row, i%n_covariates_per_row]
    sns.boxplot(x="type", y="coefficient", data= used_dataframe, hue="Split", ax = ax, palette="rocket", legend =legend)
    #chnage legend position to be right outside axis
    if legend:
        ax.legend(title="Split", loc=(1.01, 0.))

    ax.set_xlabel("")
    if i%n_covariates_per_row > 0:
        ax.set_ylabel("")
    else:
        ax.set_ylabel("Coefficient value", fontsize = 12)
    ax.set_title(covariates[i], font ="serif", fontweight ="bold", fontsize = 12)

fig.tight_layout()
fig.savefig(output_dir+f"/pho_beta_coefficients_boxplots_top{read_top_n}_{model_name}_split1to5_insurance.pdf")

In [ ]:
# we add the linear results to the lipschitz performance dicts
linear_model_test_loglike = -all_metrics_df["test loglike_traditional"].values
lipschitz_test_performance_dict[0] = linear_model_test_loglike.tolist()
fig, ax = plt.subplots(figsize=(4.5, 5.25))
boxplot = sns.boxplot(data=lipschitz_test_performance_dict, orient="h", color= "white", ax= ax)
#we round all labels to 2 digits
y_ticklabels = [f"{max(0.,np.round(float(tick.get_text())-1, 2)):.2f}" for tick in boxplot.get_yticklabels()]
# we set the first label to "GLM"
y_ticklabels[0] = "GLM"

boxplot.set_yticklabels(y_ticklabels, horizontalalignment="right")
y_ticklabels[0] = "GLM"
boxplot.set_ylabel("Allowed Lipschitz bound $L_p^b$", fontsize = 14)
boxplot.set_xlabel("Test negative Log-Likelihood", fontsize = 14)

plt.tight_layout()
boxplot.figure.savefig(output_dir+f"/lipschitz_test_performance_boxplots_insurance_{model_name}.pdf")